In [ ]:
from dataclasses import dataclass, asdict

import torch as t
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Categorical

import numpy as np
from tqdm import tqdm
from aim import Run

from mpe2 import simple_spread_v3

from agent import Dimensions
from orchestrator import TarMAC

t.set_num_threads(1)          # env stepping is Python; stop thread thrashing on tiny ops
device = "cuda"

In [ ]:
@dataclass
class Parameters:
  gamma: float          # discount factor
  gae_lambda: float     # GAE
  clip_coef: float      # PPO surrogate clip
  ent_coef: float
  vf_coef: float
  max_grad_norm: float
  update_epochs: int
  num_minibatches: int  # minibatches are over ENVS (sequences stay intact)
  lr: float = 7e-4

In [ ]:
class MPEVecEnv:
  def __init__(self, num_envs: int, n_agents: int, max_cycles: int, seed: int):
    self.envs = [
      simple_spread_v3.parallel_env(N=n_agents, max_cycles=max_cycles, continuous_actions=False)
      for _ in range(num_envs)
    ]
    self.num_envs = num_envs
    self.seeds = [seed + i for i in range(num_envs)]

    self.envs[0].reset(seed=self.seeds[0])
    self.agent_names = list(self.envs[0].agents)
    self.n_agents = len(self.agent_names)
    self.obs_dim = self.envs[0].observation_space(self.agent_names[0]).shape[0]
    self.n_actions = self.envs[0].action_space(self.agent_names[0]).n

    self._ep_returns = np.zeros(num_envs)

  def _stack(self, obs: dict) -> np.ndarray:
    return np.stack([obs[a] for a in self.agent_names]).astype(np.float32)

  def reset(self) -> np.ndarray:
    out = []
    for i, env in enumerate(self.envs):
      obs, _ = env.reset(seed=self.seeds[i])
      out.append(self._stack(obs))
    self._ep_returns[:] = 0.0
    return np.stack(out)  # (num_envs, n_agents, obs_dim)

  def step(self, actions: np.ndarray):  # (num_envs, n_agents) int
    obs_out = np.zeros((self.num_envs, self.n_agents, self.obs_dim), dtype=np.float32)
    rew_out = np.zeros((self.num_envs, self.n_agents), dtype=np.float32)
    done_out = np.zeros(self.num_envs, dtype=np.float32)
    finished = []  # episode returns that completed this step

    for i, env in enumerate(self.envs):
      act = {a: int(actions[i, j]) for j, a in enumerate(self.agent_names)}
      obs, rew, term, trunc, _ = env.step(act)
      rew_out[i] = [rew[a] for a in self.agent_names]
      self._ep_returns[i] += rew_out[i].sum()

      if not env.agents:  # episode over -> auto-reset
        done_out[i] = 1.0
        finished.append(self._ep_returns[i])
        self._ep_returns[i] = 0.0
        obs, _ = env.reset()

      obs_out[i] = self._stack(obs)

    return obs_out, rew_out, done_out, finished

In [ ]:
params = Parameters(
  gamma=0.99,
  gae_lambda=0.95,
  clip_coef=0.1,
  ent_coef=0.01,
  vf_coef=0.5,
  max_grad_norm=0.5,
  update_epochs=2,
  num_minibatches=2,
  lr=7e-4,
)

n_agents = 3
num_envs = 32
num_steps = 64       # rollout length per env
max_cycles = 25
total_timesteps = 5_000_000
seed = 1
disable_comm = False  # True -> zero messages: independent recurrent PPO baseline

t.manual_seed(seed)
np.random.seed(seed)

envs = MPEVecEnv(num_envs, n_agents, max_cycles, seed)

dims = Dimensions(
  state_dim=envs.obs_dim,
  n_actions=envs.n_actions,
  key_dim=16,
  value_dim=32,
  gru_input_dim=32,
  gru_hidden_dim=128,
)

model = TarMAC(n_agents, dims)
critic = nn.Linear(dims.gru_hidden_dim, 1)
optimizer = t.optim.Adam(list(model.parameters()) + list(critic.parameters()), params.lr, eps=1e-5)

model = TarMAC(n_agents, dims).to(device)
critic = nn.Linear(dims.gru_hidden_dim, 1).to(device)

batch_size = num_envs * num_steps
num_iterations = total_timesteps // batch_size
assert num_envs % params.num_minibatches == 0

In [ ]:
def run_sequence(
  model: TarMAC,
  critic: nn.Linear,
  obs_seq: t.Tensor,    # (T, num_envs, n_agents, obs_dim) — T=1 during rollout
  done_seq: t.Tensor,   # (T, num_envs); done[t]=1 means episode ended BEFORE obs[t]
  h: t.Tensor,          # (1, num_envs*n_agents, gru_hidden_dim)
  c: t.Tensor,          # (num_envs, n_agents, value_dim)
):
  T, n_envs, n_agents, _ = obs_seq.shape
  logits_seq, values_seq = [], []

  for step in range(T):
    d = done_seq[step]
    # reset recurrent state at episode boundaries (both carries)
    h = (1.0 - d).repeat_interleave(n_agents).view(1, -1, 1) * h
    c = (1.0 - d).view(-1, 1, 1) * c
    if disable_comm:
      c = t.zeros_like(c)

    logits, hidden, c, h = model(obs_seq[step], c, h)
    logits_seq.append(logits)
    values_seq.append(critic(hidden).squeeze(-1))

  logits_seq = t.stack(logits_seq)   # (T, num_envs, n_agents, n_actions)
  values_seq = t.stack(values_seq)   # (T, num_envs, n_agents)
  return logits_seq, values_seq, h, c

In [ ]:
def compute_gae(rewards, values, dones, next_value, next_done):
  # rewards/values: (T, num_envs, n_agents); dones: (T, num_envs)
  T = rewards.shape[0]
  advantages = t.zeros_like(rewards)
  lastgaelam = 0
  for step in reversed(range(T)):
    if step == T - 1:
      nextnonterminal = (1.0 - next_done).unsqueeze(-1)  # broadcast over agents
      nextvalues = next_value
    else:
      nextnonterminal = (1.0 - dones[step + 1]).unsqueeze(-1)
      nextvalues = values[step + 1]
    delta = rewards[step] + params.gamma * nextvalues * nextnonterminal - values[step]
    advantages[step] = lastgaelam = delta + params.gamma * params.gae_lambda * nextnonterminal * lastgaelam
  return advantages


def ppo_update(model, critic, optimizer, rollout: dict) -> dict:
  obs, actions, logprobs = rollout["obs"], rollout["actions"], rollout["logprobs"]
  dones, values = rollout["dones"], rollout["values"]
  advantages, returns = rollout["advantages"], rollout["returns"]
  h0, c0 = rollout["h0"], rollout["c0"]  # recurrent state at rollout start

  n_envs = obs.shape[1]
  envsperbatch = n_envs // params.num_minibatches
  envinds = np.arange(n_envs)
  h0_by_env = h0.reshape(1, n_envs, envs.n_agents, -1)

  stats = {}
  for epoch in range(params.update_epochs):
    np.random.shuffle(envinds)
    for start in range(0, n_envs, envsperbatch):
      mb = envinds[start:start + envsperbatch]

      # replay full sequences for this env subset, WITH grad
      mb_h0 = h0_by_env[:, mb].reshape(1, envsperbatch * envs.n_agents, -1)
      new_logits, new_values, _, _ = run_sequence(model, critic, obs[:, mb], dones[:, mb], mb_h0, c0[mb])

      dist = Categorical(logits=new_logits)
      new_logprobs = dist.log_prob(actions[:, mb])
      entropy = dist.entropy()

      logratio = new_logprobs - logprobs[:, mb]
      ratio = logratio.exp()

      mb_adv = advantages[:, mb]
      mb_adv = (mb_adv - mb_adv.mean()) / (mb_adv.std() + 1e-8)

      # policy loss (clipped surrogate)
      pg_loss1 = -mb_adv * ratio
      pg_loss2 = -mb_adv * t.clamp(ratio, 1 - params.clip_coef, 1 + params.clip_coef)
      pg_loss = t.max(pg_loss1, pg_loss2).mean()

      # value loss (clipped, as in reference)
      v_unclipped = (new_values - returns[:, mb]) ** 2
      v_clipped = values[:, mb] + t.clamp(new_values - values[:, mb], -params.clip_coef, params.clip_coef)
      v_loss = 0.5 * t.max(v_unclipped, (v_clipped - returns[:, mb]) ** 2).mean()

      entropy_loss = entropy.mean()
      loss = pg_loss - params.ent_coef * entropy_loss + params.vf_coef * v_loss

      optimizer.zero_grad()
      loss.backward()
      nn.utils.clip_grad_norm_(list(model.parameters()) + list(critic.parameters()), params.max_grad_norm)
      optimizer.step()

      with t.no_grad():
        stats = {
          "policy_loss": pg_loss.item(),
          "value_loss": v_loss.item(),
          "entropy": entropy_loss.item(),
          "approx_kl": ((ratio - 1) - logratio).mean().item(),
        }
  return stats

In [ ]:
run = Run(experiment="tarmac_ppo")
run["hparams"] = {
  **asdict(params),
  "n_agents": n_agents,
  "num_envs": num_envs,
  "num_steps": num_steps,
  "max_cycles": max_cycles,
  "total_timesteps": total_timesteps,
  "disable_comm": disable_comm,
  "env": "simple_spread_v3",
}
print(f"aim run hash: {run.hash}")

In [ ]:
next_obs = t.as_tensor(envs.reset(), device=device)
next_done = t.zeros(num_envs, device=device)
c_next, h_next = model.initial_state(num_envs)
c_next, h_next = c_next.to(device), h_next.to(device)

global_step = 0
episode_returns = []

for iteration in tqdm(range(num_iterations)):
  # lr annealing
  frac = 1.0 - iteration / num_iterations
  optimizer.param_groups[0]["lr"] = frac * params.lr

  # --- collect rollout ---
  obs = t.zeros(num_steps, num_envs, envs.n_agents, envs.obs_dim, device=device)
  actions = t.zeros(num_steps, num_envs, envs.n_agents, dtype=t.long, device=device)
  logprobs = t.zeros(num_steps, num_envs, envs.n_agents, device=device)
  rewards = t.zeros(num_steps, num_envs, envs.n_agents, device=device)
  dones = t.zeros(num_steps, num_envs, device=device)
  values = t.zeros(num_steps, num_envs, envs.n_agents, device=device)

  h0, c0 = h_next.clone(), c_next.clone()  # state at rollout start, for replay

  for step in range(num_steps):
    global_step += num_envs
    obs[step] = next_obs
    dones[step] = next_done

    with t.no_grad():
      logits, value, h_next, c_next = run_sequence(
        model, critic, next_obs.unsqueeze(0), next_done.unsqueeze(0), h_next, c_next
      )
      dist = Categorical(logits=logits.squeeze(0))
      action = dist.sample()
      logprobs[step] = dist.log_prob(action)
      values[step] = value.squeeze(0)
      actions[step] = action

    next_obs_np, reward, done_np, finished = envs.step(action.cpu().numpy())
    rewards[step] = t.as_tensor(reward, device=device)
    next_obs = t.as_tensor(next_obs_np, device=device)
    next_done = t.as_tensor(done_np, dtype=t.float32, device=device)

    for ep_ret in finished:
      episode_returns.append(ep_ret)
      run.track(ep_ret, name="episode_return", step=global_step)

  # --- GAE ---
  with t.no_grad():
    _, next_value, _, _ = run_sequence(
      model, critic, next_obs.unsqueeze(0), next_done.unsqueeze(0), h_next, c_next
    )
    advantages = compute_gae(rewards, values, dones, next_value.squeeze(0), next_done)
    returns = advantages + values

  # --- update ---
  rollout = dict(obs=obs, actions=actions, logprobs=logprobs, dones=dones,
                 values=values, advantages=advantages, returns=returns, h0=h0, c0=c0)
  stats = ppo_update(model, critic, optimizer, rollout)

  for name, val in stats.items():
    run.track(val, name=name, step=global_step)

checkpoint = {
  "model": model.state_dict(),
  "critic": critic.state_dict(),
  "optimizer": optimizer.state_dict(),
}
t.save(checkpoint, "checkpoints/tarmac_ppo_spread.pth")

In [ ]:
import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Scatter(
  y=episode_returns,
  mode='lines',
  name='Raw',
  line=dict(color='steelblue')
))

fig.update_layout(
  title='Training Episode Returns',
  xaxis_title='Episode',
  yaxis_title='Total Team Reward',
)

fig.show()

In [ ]:
ckpt = t.load("checkpoints/tarmac_ppo_spread.pth", map_location=device)
model.load_state_dict(ckpt["model"])

env_eval = simple_spread_v3.parallel_env(N=n_agents, max_cycles=max_cycles,
                                         continuous_actions=False, render_mode="human")
observations, _ = env_eval.reset(seed=0)
agent_names = list(env_eval.agents)

c, h = model.initial_state(1)
c, h = c.to(device), h.to(device)
eval_reward = 0.0

import time
with t.no_grad():
  while env_eval.agents:
    obs_ = t.stack([t.as_tensor(observations[a], dtype=t.float32) for a in agent_names]).unsqueeze(0).to(device)
    logits, _, c, h = model(obs_, c, h)
    action = Categorical(logits=logits.squeeze(0)).sample().cpu()

    observations, rew, _, _, _ = env_eval.step({a: int(action[i]) for i, a in enumerate(agent_names)})
    env_eval.render()
    time.sleep(0.1)
    eval_reward += sum(rew.values())

env_eval.close()
eval_reward